# Archetypes in PermCell Signature Space

Instead of deconvolving cells over **markers**, this notebook deconvolves them over **PermCell
signature scores** — so each archetype is described directly in terms of biological programmes
(luminal, EMT, Polycomb repression, proliferation, …) rather than individual antibodies.

The pipeline deliberately mirrors `CellLines_CyEmbed_PerLine_Matching.ipynb` step for step, so the
two are directly comparable: per-line independent fits → K selection → cross-line matching →
meta-archetypes.

### Key choices

1. **Raw, unsmoothed scores.** Scores come from calling `sipsic_like_scores_v3` directly rather than
   `Run.run_permcell`, which always applies Gaussian smoothing first. Beyond saving the work, this
   removes a real circularity: smoothing over a UMAP built from the same markers makes scores
   spatially autocorrelated, while PermCell's null permutes *markers*, not *positions*, so it cannot
   absorb that. Raw scores keep the null valid.

2. **Z is the score** (not `Zdir` or `Zabs`) — the permutation z-statistic of the observed weighted
   score against size- and weight-matched random marker sets.

3. **Input is the normalised layer, z-scored.** `norm_divide`, standardised with a scaler fitted once
   across all five lines. The z-scoring is required, not cosmetic: `sipsic_like_scores_v3`
   row-centres each cell across markers, which is only meaningful when markers share a scale.

4. **15 discovery signatures.** The full authored set was 22. Seven are excluded from discovery:
   three composites that were written *from* findings in this dataset (circular), and four that are
   near-mirror images of retained sets (`EMT` vs `Epithelial_adhesion`, r = −0.97; `Mitotic` vs
   `Quiescent`, r = −0.93). Near-collinear inputs distort archetype geometry the same way the global
   chromatin axis did in marker space. All seven are scored post-hoc instead, and the dropped mirrors
   are exactly recoverable as negatives of what was kept.

5. **No sample offset**, and a **shared scaler** across lines — identical reasoning to the marker-space
   notebook.

## 0. Setup

In [ ]:
import sys, json, itertools
from pathlib import Path

CE_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Experiments/CyEmbed"
HELPER = "/Users/ronguy/Dropbox/Work/CyTOF/HelperPackage"
SCRIPTS = "/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina/scripts"
for p in (CE_PATH, HELPER, SCRIPTS):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
from scipy import stats
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

from CyEmbed.data import extract_matrix, fit_scaler, preprocess_array, split_train_val_indices
from CyEmbed.train import build_sweep_configs, run_sweep
from CyEmbed.analysis import load_run_outputs
from permcell_signatures import SIGNATURES, SIGNATURES_WITH_POSTHOC, COMBINED_POSTHOC, MIRRORED_POSTHOC

plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 10, "ytick.labelsize": 10})

BASE = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
PLOTS = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)
SCORES = BASE / "outputs/permcell_scores"
SWEEP = BASE / "outputs/cyembed_signature_sweep"
MARKER_SWEEP = BASE / "outputs/cyembed_perline_sweep"

LINES = ["MDAMB468", "HCC70", "SUM149", "HCC1937", "MCF7"]
LINE_DISP = {"MDAMB468": "MDA-MB-468", "HCC70": "HCC70", "SUM149": "SUM149",
             "HCC1937": "HCC1937", "MCF7": "MCF7"}
DISP_ORDER = ["HCC1937", "HCC70", "MCF7", "MDA-MB-468", "SUM149"]
LINE_COLORS = dict(zip(DISP_ORDER, ["#66c2a5", "#8da0cb", "#a6d854", "#fc8d62", "#b3b3b3"]))

SEED = 42
K_RANGE = list(range(2, 13))
SEEDS = [42, 1, 2]
FAILURE_TOLERANCE = 0.05

BASE_CONFIG = dict(
    model_type="deterministic", decoder_type="factorized",
    d=16, hidden_dims=[64, 32], tau=1.0,
    epochs=1500, early_stopping=True, patience=20,
    min_delta=0.0, restore_best_weights=True,
    lr=1e-3, batch_size=2048, weight_decay=1e-5, dropout=0.0,
    logit_normalizer="entmax", entmax_alpha=1.5, grad_clip_norm=5.0,
    separation_mode="cosine_sq", balance_mode="l2_uniform",
    lambda_entropy=1e-3, lambda_sep=1e-3, lambda_balance=0.05,
    recon_loss_type="mse", device="cpu", deterministic=True, seed=SEED,
)
print(f"{len(SIGNATURES)} discovery signatures | "
      f"{len(SIGNATURES_WITH_POSTHOC) - len(SIGNATURES)} post-hoc")

## 1. PermCell scores

Computed by `scripts/permcell_scores.py` (rerun it to regenerate). Loaded here along with the shared
scaler that puts every line's signature axes on the same scale.

In [ ]:
zs = {line: pd.read_csv(SCORES / f"{line}_Z.csv") for line in LINES}
SIG_NAMES = list(zs[LINES[0]].columns)
assert all(list(df.columns) == SIG_NAMES for df in zs.values()), "signature order mismatch"

stacked = np.vstack([zs[l].to_numpy(np.float32) for l in LINES])
sample_ids_all = np.concatenate([np.repeat(LINE_DISP[l], len(zs[l])) for l in LINES])
combined = ad.AnnData(X=stacked,
                      obs=pd.DataFrame({"cell_line": sample_ids_all},
                                       index=[str(i) for i in range(len(stacked))]),
                      var=pd.DataFrame(index=SIG_NAMES))
bundle_all = extract_matrix(adata=combined, layer=None, sample_col="cell_line")
SHARED_SCALER, _ = fit_scaler(bundle_all.X, mode="zscore",
                              sample_ids=bundle_all.sample_ids, balanced_max_per_sample=5000)

line_data = {}
for line in LINES:
    df = zs[line]
    a = ad.AnnData(X=df.to_numpy(np.float32),
                   obs=pd.DataFrame({"cell_line": LINE_DISP[line]},
                                    index=[f"{line}_{i}" for i in range(len(df))]),
                   var=pd.DataFrame(index=SIG_NAMES))
    b = extract_matrix(adata=a, layer=None, sample_col="cell_line")
    x = preprocess_array(b.X, SHARED_SCALER)
    tr, va = split_train_val_indices(n_cells=len(x), val_fraction=0.2, seed=SEED,
                                     stratify_labels=None)
    line_data[line] = dict(X=x, Z=df, bundle=b, train_idx=tr, val_idx=va)

print(f"{len(SIG_NAMES)} signatures x {len(stacked):,} cells total\n")
for line in LINES:
    print(f"  {LINE_DISP[line]:<11} {line_data[line]['X'].shape[0]:>7,} cells")

print("\n=== Signature score distributions (pooled, raw Z before scaling) ===")
pooled = pd.DataFrame(stacked, columns=SIG_NAMES)
print(pooled.describe().loc[["mean", "std", "min", "max"]].round(2).to_string())

In [ ]:
# Redundancy of the discovery space -- the justification for pruning to 15
C = pooled.corr()
A = C.abs().values.copy(); np.fill_diagonal(A, 0)
ev = np.linalg.eigvalsh(np.corrcoef(stacked.T))[::-1]; ev = ev[ev > 0]
print(f"max pairwise |r| among discovery signatures : {A.max():.3f}")
print(f"effective dimensionality (participation ratio): {ev.sum()**2/(ev**2).sum():.2f} of {len(SIG_NAMES)}")
print(f"variance in first 5 PCs: {np.round(100*ev[:5]/ev.sum(), 1)}")

fig, ax = plt.subplots(figsize=(9.5, 8))
order = C.abs().sum().sort_values(ascending=False).index
sns.heatmap(C.loc[order, order], cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            annot=True, fmt=".2f", annot_kws={"size": 6.5},
            cbar_kws={"label": "Pearson r", "shrink": 0.7}, ax=ax)
ax.set_title("PermCell discovery-signature correlation (all 354k cells)\n"
             "mirror pairs already removed; max |r| = "
             f"{A.max():.2f}", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(PLOTS / "PermCell_Signature_Correlation.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Per-line sweeps in signature space

`K = 2..12` × seeds {42, 1, 2}, independently per line. Completed runs are reused by fingerprint.
Equivalent script: `scripts/cyembed_signature_sweep.py --line all`.

In [ ]:
configs = build_sweep_configs({"K": K_RANGE, "seed": SEEDS})
for line in LINES:
    out_dir = SWEEP / line
    if out_dir.exists():
        stale = [d for d in out_dir.glob("run_*") if d.is_dir() and not any(d.iterdir())]
        for d in stale:
            d.rmdir()
    ld = line_data[line]
    run_sweep(x=ld["X"], marker_names=list(ld["bundle"].marker_names),
              cell_ids=list(ld["bundle"].cell_ids), output_root=out_dir,
              base_config=BASE_CONFIG, sweep_configs=configs,
              train_idx=ld["train_idx"], val_idx=ld["val_idx"],
              sample_ids=None, scaler_state=SHARED_SCALER.to_dict())

rows = []
for line in LINES:
    for r in sorted((SWEEP / line).glob("run_*")):
        sm_f, cf_f = r / "summary_metrics.json", r / "config.json"
        if not (sm_f.exists() and cf_f.exists()):
            continue
        sm, cfg = json.loads(sm_f.read_text()), json.loads(cf_f.read_text())
        rows.append({"line": LINE_DISP[line], "K": cfg["K"], "seed": cfg["seed"],
                     "val_recon": sm.get("best_val_recon", sm.get("val", {}).get("recon_mse")),
                     "run_dir": str(r)})
runs_df = pd.DataFrame(rows)

want = {(LINE_DISP[l], k, s) for l in LINES for k in K_RANGE for s in SEEDS}
have = {(r.line, r.K, r.seed) for r in runs_df.itertuples()}
missing = sorted(want - have)
print(f"runs: {len(runs_df)}/{len(want)}")
print("grid complete" if not missing else f"!! MISSING {len(missing)}: {missing[:8]}")

## 3. Choosing $K$ per line

Same rule as the marker-space analysis: seeds scoring >5% above the best seed at a given $K$ are
optimiser failures and are excluded from the error curve; Kneedle elbow on the best-of-seeds curve;
selected $K$ = largest $K$ at or below the elbow where **every** seed converged.

In [ ]:
def kneedle_elbow(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x) < 3:
        return x[0] if len(x) else None
    xn = (x - x.min()) / (np.ptp(x) + 1e-12)
    yn = (y - y.min()) / (np.ptp(y) + 1e-12)
    chord = yn[0] + (xn - xn[0]) * (yn[-1] - yn[0]) / (xn[-1] - xn[0] + 1e-12)
    return x[int(np.argmax(chord - yn))]


def matched_cosine(a1, a2):
    n1 = a1 / (np.linalg.norm(a1, axis=1, keepdims=True) + 1e-12)
    n2 = a2 / (np.linalg.norm(a2, axis=1, keepdims=True) + 1e-12)
    sim = n1 @ n2.T
    r, c = linear_sum_assignment(-sim)
    return float(sim[r, c].mean())


sel_rows, failures = [], []
for line_disp, g_line in runs_df.groupby("line"):
    g_line = g_line.copy()
    g_line["best_at_K"] = g_line.groupby("K")["val_recon"].transform("min")
    g_line["excess"] = g_line["val_recon"] / g_line["best_at_K"] - 1.0
    g_line["converged"] = g_line["excess"] <= FAILURE_TOLERANCE
    failures.append(g_line[~g_line["converged"]])
    per_K = []
    for k, g in g_line.groupby("K"):
        A = {int(r.seed): np.load(Path(r.run_dir) / "A_hat.npy") for r in g.itertuples()}
        sd = sorted(A)
        sims = [matched_cosine(A[i], A[j]) for ii, i in enumerate(sd) for j in sd[ii + 1:]]
        per_K.append({"line": line_disp, "K": k, "n_seeds": len(g),
                      "n_converged": int(g["converged"].sum()),
                      "val_best": g["val_recon"].min(),
                      "stability": float(np.mean(sims)) if sims else np.nan,
                      "_A": A, "_g": g})
    per_K = pd.DataFrame(per_K).sort_values("K").reset_index(drop=True)
    elbow = kneedle_elbow(per_K["K"].values, per_K["val_best"].values)
    ok = per_K[(per_K["K"] <= elbow) & (per_K["n_converged"] == per_K["n_seeds"])]
    K_sel = int(ok["K"].max()) if len(ok) else int(per_K.loc[per_K["val_best"].idxmin(), "K"])
    per_K["elbow"] = elbow; per_K["K_sel"] = K_sel
    sel_rows.append(per_K)

sel_all = pd.concat(sel_rows, ignore_index=True)
failures = pd.concat(failures, ignore_index=True)

print("=== Convergence failures ===")
if len(failures) == 0:
    print("  none")
else:
    for _, r in failures.sort_values(["line", "K", "seed"]).iterrows():
        print(f"  {r['line']:<11} K={int(r['K']):<3} seed={int(r['seed']):<3} "
              f"{r['val_recon']:.5f} vs best {r['best_at_K']:.5f} (+{100*r['excess']:.1f}%)")

summary = (sel_all.groupby("line")
           .agg(elbow=("elbow", "first"), K_selected=("K_sel", "first")).reset_index())
summary["val_at_K"] = [sel_all[(sel_all.line == r.line) & (sel_all.K == r.K_selected)]["val_best"].iloc[0]
                       for r in summary.itertuples()]
summary["stability_at_K"] = [sel_all[(sel_all.line == r.line) & (sel_all.K == r.K_selected)]["stability"].iloc[0]
                             for r in summary.itertuples()]
print("\n=== Per-line K selection (SIGNATURE space) ===")
print(summary.round(4).to_string(index=False))
K_SEL = dict(zip(summary["line"], summary["K_selected"]))
print(f"\nSelected K: {K_SEL}   total archetypes = {sum(K_SEL.values())}")

## 4. Representative model per line

Medoid seed at the selected $K$. Two representations are kept: the raw fitted vertex, and the vertex
centred on that line's own cell centroid (`A_cen`), which is what cross-line matching uses — it asks
"is this the same state *relative to its line*" rather than re-discovering line identity.

In [ ]:
per_line_model = {}
for line in DISP_ORDER:
    s = sel_all[(sel_all.line == line) & (sel_all.K == K_SEL[line])].iloc[0]
    A_by_seed, g = s["_A"], s["_g"]
    conv = set(g.loc[g["converged"], "seed"].astype(int))
    cand = [sd for sd in sorted(A_by_seed) if sd in conv] or sorted(A_by_seed)
    if len(cand) > 1:
        msim = {sd: np.mean([matched_cosine(A_by_seed[sd], A_by_seed[o]) for o in cand if o != sd])
                for sd in cand}
        medoid = max(msim, key=msim.get)
    else:
        medoid, msim = cand[0], {cand[0]: np.nan}
    run_dir = g.loc[g["seed"] == medoid, "run_dir"].iloc[0]
    out = load_run_outputs(run_dir)
    lk = [k for k, v in LINE_DISP.items() if v == line][0]
    X_line = line_data[lk]["X"]
    per_line_model[line] = {
        "line_key": lk, "K": int(K_SEL[line]), "seed": int(medoid), "run_dir": run_dir,
        "A_raw": out["A_hat"], "A_cen": out["A_hat"] - X_line.mean(axis=0),
        "W": out["W"], "sigs": out["marker_names"],
    }
    print(f"{line:<11} K={K_SEL[line]:<3} medoid seed={medoid:<3} "
          f"(mean matched cosine {msim.get(medoid, float('nan')):.3f})")

assert all(per_line_model[l]["sigs"] == SIG_NAMES for l in DISP_ORDER)
TOTAL_ARCH = sum(m["K"] for m in per_line_model.values())
print(f"\n{TOTAL_ARCH} archetypes total")

## 5. Cross-line matching and meta-archetypes

Identical machinery to the marker-space notebook: cosine similarity of line-centred profiles,
calibrated against a marker(signature)-permutation null, then hierarchical clustering cut at the
null-derived threshold to form meta-archetypes.

In [ ]:
def unit(v):
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)


rows, vecs = [], []
for line in DISP_ORDER:
    m = per_line_model[line]
    for k in range(m["K"]):
        rows.append({"label": f"{line}|A{k+1}", "line": line, "k": k + 1,
                     "pool_pct": 100 * float((m["W"].argmax(1) == k).mean())})
        vecs.append(m["A_cen"][k])
arch_df = pd.DataFrame(rows)
A_CEN = np.vstack(vecs)
U = unit(A_CEN)
S = U @ U.T
same_line = arch_df["line"].values[:, None] == arch_df["line"].values[None, :]

rng = np.random.default_rng(SEED)
null_perm, null_rand = [], []
for _ in range(4000):
    i, j = rng.integers(0, len(U), 2)
    if arch_df["line"].iloc[i] == arch_df["line"].iloc[j]:
        continue
    null_perm.append(float(unit(A_CEN[i][rng.permutation(A_CEN.shape[1])]) @ U[j]))
    null_rand.append(float(unit(rng.normal(size=A_CEN.shape[1])) @ U[j]))
null_perm, null_rand = np.array(null_perm), np.array(null_rand)
MATCH_THRESHOLD = float(np.quantile(np.abs(null_perm), 0.99))
LENIENT = float(np.quantile(np.abs(null_rand), 0.99))

off = S[~same_line]
print(f"Null (permuted signatures): mean |cos| {np.abs(null_perm).mean():.3f}, 99th {MATCH_THRESHOLD:.3f}")
print(f"Null (random direction)   : mean |cos| {np.abs(null_rand).mean():.3f}, 99th {LENIENT:.3f}")
print(f"Observed cross-line: mean {off.mean():.3f}, max {off.max():.3f}, "
      f"{100*(off > MATCH_THRESHOLD).mean():.1f}% above threshold")
print("\nNOTE: with only 15 signature dimensions the permutation null is looser than in 29-marker")
print("space -- fewer dimensions means a shuffled vector aligns by chance more easily.")

pair_scores = pd.DataFrame(index=DISP_ORDER, columns=DISP_ORDER, dtype=float)
pair_records = []
for l1, l2 in itertools.combinations(DISP_ORDER, 2):
    i1 = arch_df.index[arch_df.line == l1].values
    i2 = arch_df.index[arch_df.line == l2].values
    sub = S[np.ix_(i1, i2)]
    r, c = linear_sum_assignment(-sub)
    sims = sub[r, c]
    pair_scores.loc[l1, l2] = pair_scores.loc[l2, l1] = sims.mean()
    for a, b, v in zip(r, c, sims):
        pair_records.append({"arch_a": arch_df.label[i1[a]], "arch_b": arch_df.label[i2[b]],
                             "cosine": v, "matched": v > MATCH_THRESHOLD})
pairs_df = pd.DataFrame(pair_records)
np.fill_diagonal(pair_scores.values, 1.0)
print("\n=== Mean matched cosine between line pairs ===")
print(pair_scores.astype(float).round(3).to_string())
print(f"\n{int(pairs_df['matched'].sum())}/{len(pairs_df)} optimal pairings above threshold")

In [ ]:
D = np.clip((1.0 - S + (1.0 - S).T) / 2, 0, None)
np.fill_diagonal(D, 0.0)
Zl = linkage(squareform(D, checks=False), method="average")
arch_df["meta"] = fcluster(Zl, t=1.0 - MATCH_THRESHOLD, criterion="distance")

meta_rows = []
for m, g in arch_df.groupby("meta"):
    lines_in = sorted(set(g["line"]))
    n = len(lines_in)
    kind = {5: "CONSERVED (all 5)", 4: "shared (4)", 3: "shared (3)",
            2: "restricted (2)", 1: "LINE-SPECIFIC"}[n]
    idx = g.index.values
    within = S[np.ix_(idx, idx)][np.triu_indices(len(idx), 1)]
    meta_rows.append({"meta": m, "n_arch": len(g), "n_lines": n, "kind": kind,
                      "mean_within_cos": float(within.mean()) if len(within) else np.nan,
                      "mean_pool_pct": float(g["pool_pct"].mean()),
                      "dup_line": bool(g["line"].duplicated().any()),
                      "lines": ", ".join(lines_in), "members": ", ".join(g["label"])})
meta_df = pd.DataFrame(meta_rows).sort_values(["n_lines", "mean_pool_pct"],
                                              ascending=[False, False]).reset_index(drop=True)
print(f"{TOTAL_ARCH} archetypes -> {len(meta_df)} meta-archetypes\n")
print(meta_df[["meta", "n_lines", "kind", "n_arch", "mean_within_cos",
               "mean_pool_pct", "lines"]].round(3).to_string(index=False))
print("\n=== Breakdown ===")
for kind, g in meta_df.groupby("kind", sort=False):
    print(f"  {kind:<20} {len(g)}")
dupes = meta_df[meta_df["dup_line"]]
if len(dupes):
    print(f"\n  NOTE: {len(dupes)} cluster(s) merge >1 archetype from the same line:")
    for _, r in dupes.iterrows():
        print(f"    meta {r['meta']}: {r['members']}")

In [ ]:
# What is each meta-archetype, in PROGRAMME terms? This is the payoff of signature space.
meta_prof = {}
print("=== Meta-archetype signature signatures (mean line-centred Z profile) ===")
for r in meta_df.itertuples():
    idx = arch_df.index[arch_df.meta == r.meta].values
    prof = pd.Series(A_CEN[idx].mean(0), index=SIG_NAMES)
    meta_prof[r.meta] = prof
    hi = ", ".join(f"{s}{prof[s]:+.1f}" for s in prof.sort_values(ascending=False).index[:4])
    lo = ", ".join(f"{s}{prof[s]:+.1f}" for s in prof.sort_values().index[:4])
    print(f"\nM{r.meta} [{r.kind}]  {r.mean_pool_pct:.1f}% of cells | {r.members}")
    print(f"   UP  : {hi}")
    print(f"   DOWN: {lo}")

meta_prof_df = pd.DataFrame(meta_prof).T
meta_prof_df.index = [f"M{m}" for m in meta_prof_df.index]

fig, axes = plt.subplots(1, 2, figsize=(20, 0.5 * len(meta_df) + 4),
                         gridspec_kw={"width_ratios": [1.55, 1]})
v = float(np.abs(meta_prof_df.values).max())
sns.heatmap(meta_prof_df, cmap="RdBu_r", center=0, vmin=-v, vmax=v, annot=True, fmt=".1f",
            annot_kws={"size": 7}, cbar_kws={"label": "line-centred signature Z"}, ax=axes[0])
axes[0].set_title("Meta-archetypes described by BIOLOGICAL PROGRAMME\n"
                  "(rows sorted by conservation)", fontweight="bold", pad=12)
axes[0].set_yticklabels([f"{lab} ({r.n_lines}/5)" for lab, r in
                         zip(meta_prof_df.index, meta_df.itertuples())], rotation=0)

present = pd.crosstab(arch_df["meta"], arch_df["line"]).reindex(
    columns=DISP_ORDER, fill_value=0).reindex(meta_df["meta"].values)
sns.heatmap(present.values, annot=True, fmt="d", cmap="Blues", vmin=0, vmax=2,
            xticklabels=DISP_ORDER,
            yticklabels=[f"M{r.meta} ({r.n_lines}/5)" for r in meta_df.itertuples()],
            linewidths=0.6, linecolor="white",
            cbar_kws={"label": "archetypes contributed"}, ax=axes[1])
axes[1].set_title("Which lines contribute", fontweight="bold")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.savefig(PLOTS / "PermCell_MetaArchetypes.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Post-hoc composites — derived from the result, not baked into it

The seven excluded signatures are scored here, *after* archetypes are fitted. Two uses:

- **Mirror sets** (`Epithelial_adhesion`, `Quiescent`, `Mitotic`) — sanity checks; they should track
  the negative of what was retained.
- **Composites** (`BasalStem_K9me2_slow`, `Plastic_bivalent_stem`, `Luminal_proliferative`) — these
  encode hypotheses from earlier analysis. Because they were kept out of discovery, agreement here is
  genuine corroboration rather than circularity.

Better still, composites are derived **empirically**: for each meta-archetype we read off which
discovery signatures co-occur, which is the data's own answer to "what programmes combine".

In [ ]:
# --- 6.1 Composites derived FROM the data --------------------------------------------------------
print("=== Data-derived composite phenotypes (|Z| >= 0.75 x row max) ===")
derived = {}
for r in meta_df.itertuples():
    prof = meta_prof_df.loc[f"M{r.meta}"]
    thr = 0.75 * prof.abs().max()
    up = [s for s in prof.index if prof[s] >= thr]
    dn = [s for s in prof.index if prof[s] <= -thr]
    derived[f"M{r.meta}"] = {"up": up, "down": dn}
    lbl = " + ".join(up) if up else "(none)"
    if dn:
        lbl += " - " + " - ".join(dn)
    print(f"  M{r.meta} ({r.n_lines}/5, {r.mean_pool_pct:4.0f}% cells): {lbl}")

# --- 6.2 Score the HELD-OUT signatures and test them against the fitted archetypes ---------------
import PermCell_Smooth as PCS
from cytofstandard import Project

posthoc_sets = {k: v for k, v in SIGNATURES_WITH_POSTHOC.items() if k not in SIGNATURES}
print(f"\nScoring {len(posthoc_sets)} held-out signatures (not used to fit anything)...")

# Rebuild the same z-scored marker matrices the discovery scores were computed from.
marker_adatas = {}
for line in LINES:
    a = Project.load(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{line}_NormCompare") \
               .get_run(line).read_adata()
    bio = [m for m in a.var_names if m not in ["H3", "H3.3", "H4"]]
    sub = ad.AnnData(X=a[:, bio].layers["norm_divide"].copy(),
                     obs=a.obs[["cell_uuid"]].copy(), var=a[:, bio].var.copy())
    sub.obs["cell_line"] = LINE_DISP[line]
    marker_adatas[line] = sub

mk_comb = ad.concat([marker_adatas[l] for l in LINES], join="outer")
mk_bundle = extract_matrix(adata=mk_comb, layer=None, sample_col="cell_line")
MK_SCALER, _ = fit_scaler(mk_bundle.X, mode="zscore",
                          sample_ids=mk_bundle.sample_ids, balanced_max_per_sample=5000)

posthoc_Z = {}
for line in LINES:
    b = extract_matrix(adata=marker_adatas[line], layer=None, sample_col="cell_line")
    Xm = preprocess_array(b.X, MK_SCALER).astype(np.float32)
    tmp = ad.AnnData(X=Xm, obs=marker_adatas[line].obs.copy(), var=marker_adatas[line].var.copy())
    Zp, _, _, _, _ = PCS.sipsic_like_scores_v3(
        adata=tmp, marker_sets=posthoc_sets, layer="X", n_perm=2000, seed=0,
        exclude_set=True, two_sided=True, abs_variant=True, progress=False,
        normalize_set_weights="l2", prefer_permutation=True, perm_batch=512)
    posthoc_Z[line] = Zp

# Mirror check: each held-out mirror should be strongly anti-correlated with its retained partner.
print("\n=== Mirror check (held-out vs retained, pooled across lines) ===")
mirror_partner = {"Epithelial_adhesion": "EMT", "Quiescent": "Proliferation",
                  "Mitotic": "Proliferation"}
for held, partner in mirror_partner.items():
    if held not in posthoc_Z[LINES[0]].columns:
        continue
    a_ = np.concatenate([posthoc_Z[l][held].to_numpy() for l in LINES])
    b_ = np.concatenate([zs[l][partner].to_numpy() for l in LINES])
    print(f"  {held:<24} vs {partner:<16} r = {np.corrcoef(a_, b_)[0,1]:+.3f}")

# Corroboration: do the composite scores peak in the meta-archetypes we would expect?
print("\n=== Held-out COMPOSITE scores by meta-archetype (mean Z of dominated cells) ===")
comp_names = [c for c in COMBINED_POSTHOC if c in posthoc_Z[LINES[0]].columns]
rows_c = []
for line in DISP_ORDER:
    lk = per_line_model[line]["line_key"]
    dom = per_line_model[line]["W"].argmax(1)
    for k in range(per_line_model[line]["K"]):
        lab = f"{line}|A{k+1}"
        meta = int(arch_df.loc[arch_df.label == lab, "meta"].iloc[0])
        m_ = dom == k
        if m_.sum() < 50:
            continue
        rec = {"meta": f"M{meta}", "arch": lab, "n": int(m_.sum())}
        for c in comp_names:
            rec[c] = float(posthoc_Z[lk][c].to_numpy()[m_].mean())
        rows_c.append(rec)
comp_tbl = pd.DataFrame(rows_c).groupby("meta")[comp_names].mean()
comp_tbl = comp_tbl.reindex([f"M{m}" for m in meta_df["meta"]])
print(comp_tbl.round(2).to_string())
for c in comp_names:
    print(f"  {c:<24} peaks in {comp_tbl[c].idxmax()}  (Z = {comp_tbl[c].max():+.2f})")
print("\nThese signatures were held out of discovery, so where they peak is corroboration,")
print("not circularity.")

## 7. Signature space vs marker space

The same five lines were deconvolved over 29 markers in
`CellLines_CyEmbed_PerLine_Matching.ipynb`. Comparing the two answers a concrete question: does
scoring programmes first change **what** structure is found, or only how it is described?

Three comparisons:
1. **How many archetypes** each line needs.
2. **Cross-seed stability** — signature space has fewer, less redundant dimensions, so archetypes
   should be more reproducible if the representation is better conditioned.
3. **Conservation** — marker space found only 3 of 11 meta-archetypes shared across ≥4 lines, and two
   of those were poles of the global chromatin magnitude axis. Signature space removes that axis by
   construction (PermCell row-centres), so conservation should be *more* biological here.

In [ ]:
# Marker-space results for comparison
mk_rows = []
for line in LINES:
    for r in sorted((MARKER_SWEEP / line).glob("run_*")):
        sm_f, cf_f = r / "summary_metrics.json", r / "config.json"
        if not (sm_f.exists() and cf_f.exists()):
            continue
        sm, cfg = json.loads(sm_f.read_text()), json.loads(cf_f.read_text())
        mk_rows.append({"line": LINE_DISP[line], "K": cfg["K"], "seed": cfg["seed"],
                        "val_recon": sm.get("best_val_recon", sm.get("val", {}).get("recon_mse")),
                        "run_dir": str(r)})
mk_df = pd.DataFrame(mk_rows)

mk_sel = []
for line_disp, g_line in mk_df.groupby("line"):
    g_line = g_line.copy()
    g_line["best_at_K"] = g_line.groupby("K")["val_recon"].transform("min")
    g_line["converged"] = (g_line["val_recon"] / g_line["best_at_K"] - 1.0) <= FAILURE_TOLERANCE
    per_K = []
    for k, g in g_line.groupby("K"):
        A = {int(r.seed): np.load(Path(r.run_dir) / "A_hat.npy") for r in g.itertuples()}
        sd = sorted(A)
        sims = [matched_cosine(A[i], A[j]) for ii, i in enumerate(sd) for j in sd[ii + 1:]]
        per_K.append({"line": line_disp, "K": k, "n_seeds": len(g),
                      "n_converged": int(g["converged"].sum()),
                      "val_best": g["val_recon"].min(),
                      "stability": float(np.mean(sims)) if sims else np.nan})
    per_K = pd.DataFrame(per_K).sort_values("K")
    elbow = kneedle_elbow(per_K["K"].values, per_K["val_best"].values)
    ok = per_K[(per_K["K"] <= elbow) & (per_K["n_converged"] == per_K["n_seeds"])]
    Ks = int(ok["K"].max()) if len(ok) else int(per_K.loc[per_K["val_best"].idxmin(), "K"])
    mk_sel.append({"line": line_disp, "K_marker": Ks,
                   "stability_marker": float(per_K.loc[per_K.K == Ks, "stability"].iloc[0])})
mk_sel = pd.DataFrame(mk_sel)

cmp_tbl = summary[["line", "K_selected", "stability_at_K"]].rename(
    columns={"K_selected": "K_signature", "stability_at_K": "stability_signature"}
).merge(mk_sel, on="line")
cmp_tbl = cmp_tbl[["line", "K_marker", "K_signature", "stability_marker", "stability_signature"]]
print("=== Signature space vs marker space, per line ===")
print(cmp_tbl.round(3).to_string(index=False))
print(f"\nmean cross-seed stability: marker {cmp_tbl['stability_marker'].mean():.3f} "
      f"vs signature {cmp_tbl['stability_signature'].mean():.3f}")
print(f"total archetypes: marker {cmp_tbl['K_marker'].sum()} vs signature {cmp_tbl['K_signature'].sum()}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(cmp_tbl)); w = 0.38
axes[0].bar(x - w/2, cmp_tbl["K_marker"], w, label="marker space (29)", color="#8da0cb",
            edgecolor="black")
axes[0].bar(x + w/2, cmp_tbl["K_signature"], w, label="signature space (15)", color="#fc8d62",
            edgecolor="black")
axes[0].set_xticks(x); axes[0].set_xticklabels(cmp_tbl["line"], rotation=20)
axes[0].set_ylabel("selected K"); axes[0].set_title("Archetypes needed per line", fontweight="bold")
axes[0].legend(fontsize=9); axes[0].grid(True, axis="y", linestyle="--", alpha=0.4)

axes[1].bar(x - w/2, cmp_tbl["stability_marker"], w, label="marker", color="#8da0cb",
            edgecolor="black")
axes[1].bar(x + w/2, cmp_tbl["stability_signature"], w, label="signature", color="#fc8d62",
            edgecolor="black")
axes[1].set_xticks(x); axes[1].set_xticklabels(cmp_tbl["line"], rotation=20)
axes[1].set_ylim(0, 1.02); axes[1].set_ylabel("cross-seed matched cosine")
axes[1].set_title("Archetype reproducibility", fontweight="bold")
axes[1].legend(fontsize=9); axes[1].grid(True, axis="y", linestyle="--", alpha=0.4)

counts = meta_df["kind"].value_counts()
order_k = ["CONSERVED (all 5)", "shared (4)", "shared (3)", "restricted (2)", "LINE-SPECIFIC"]
vals = [counts.get(k, 0) for k in order_k]
cols = ["#1a9850", "#91cf60", "#d9ef8b", "#fee08b", "#d73027"]
axes[2].bar(range(len(order_k)), vals, color=cols, edgecolor="black")
axes[2].set_xticks(range(len(order_k)))
axes[2].set_xticklabels([k.split(" (")[0] for k in order_k], rotation=25, ha="right")
axes[2].set_ylabel("meta-archetypes")
axes[2].set_title(f"Conservation in signature space\n({len(meta_df)} meta-archetypes)",
                  fontweight="bold")
axes[2].grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS / "PermCell_vs_Marker_Space.png", dpi=200, bbox_inches="tight")
plt.show()

## 7b. Identifying every meta-archetype

Each meta-archetype is described three ways so the label is not resting on one view:

* **Signature profile** — the line-centred PermCell Z the archetypes were actually fitted on.
* **Marker profile** — the mean observed z-scored marker values of the cells it dominates. This is
  the grounding check: a state called "EMT" should be Vimentin-high in raw marker terms too.
* **Occupancy** — how many cells and which lines.

The marker view matters because signature scores are *compositional* (PermCell row-centres each
cell), so a high signature Z means "this programme dominates this cell's profile", not "these markers
are absolute-high". Reading both together is what makes a label defensible.

In [ ]:
# --- 7b.1 Identification table: signatures + markers + occupancy -------------------------------
from cytofstandard import Project

# Marker-space matrices for the same cells, in the same order, for grounding.
marker_X, MARKERS_LIST = {}, None
for line in LINES:
    a = Project.load(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{line}_NormCompare") \
               .get_run(line).read_adata()
    bio = [m for m in a.var_names if m not in ["H3", "H3.3", "H4"]]
    sub = ad.AnnData(X=a[:, bio].layers["norm_divide"].copy(),
                     obs=a.obs[["cell_uuid"]].copy(), var=a[:, bio].var.copy())
    sub.obs["cell_line"] = LINE_DISP[line]
    marker_adatas_id = sub
    if MARKERS_LIST is None:
        MARKERS_LIST = list(sub.var_names)
    marker_X[LINE_DISP[line]] = sub

mk_comb_id = ad.concat([marker_X[LINE_DISP[l]] for l in LINES], join="outer")
mk_b = extract_matrix(adata=mk_comb_id, layer=None, sample_col="cell_line")
MK_SCALER_ID, _ = fit_scaler(mk_b.X, mode="zscore",
                             sample_ids=mk_b.sample_ids, balanced_max_per_sample=5000)
MK = {}
for line in LINES:
    b = extract_matrix(adata=marker_X[LINE_DISP[line]], layer=None, sample_col="cell_line")
    MK[LINE_DISP[line]] = preprocess_array(b.X, MK_SCALER_ID)

# Per-line dominant-archetype assignment
dom_by_line = {l: per_line_model[l]["W"].argmax(1) for l in DISP_ORDER}

ident_rows = []
for r in meta_df.itertuples():
    g = arch_df[arch_df.meta == r.meta]
    sigp = pd.Series(A_CEN[g.index.values].mean(0), index=SIG_NAMES)
    tot, acc = 0, []
    for _, a_ in g.iterrows():
        sel = dom_by_line[a_.line] == (a_.k - 1)
        n = int(sel.sum()); tot += n
        acc.append(MK[a_.line][sel].mean(0) * n)
    mkp = pd.Series(np.sum(acc, axis=0) / max(tot, 1), index=MARKERS_LIST)
    ident_rows.append({
        "meta": f"M{r.meta}", "n_lines": r.n_lines, "n_cells": tot,
        "pct_of_line": r.mean_pool_pct, "lines": r.lines,
        "sig_up": "; ".join(f"{s}{sigp[s]:+.1f}" for s in sigp.sort_values(ascending=False).index[:3]),
        "sig_dn": "; ".join(f"{s}{sigp[s]:+.1f}" for s in sigp.sort_values().index[:3]),
        "mk_up": "; ".join(f"{s}{mkp[s]:+.2f}" for s in mkp.sort_values(ascending=False).index[:4]),
        "mk_dn": "; ".join(f"{s}{mkp[s]:+.2f}" for s in mkp.sort_values().index[:4]),
        "KI67": mkp["KI67"], "members": "; ".join(g["label"]),
        "_sig": sigp, "_mk": mkp,
    })
ident = pd.DataFrame(ident_rows)

print("=" * 100)
print("META-ARCHETYPE IDENTIFICATION")
print("=" * 100)
for r in ident.itertuples():
    print(f"\n{r.meta}  [{r.n_lines}/5 lines]  {r.n_cells:,} cells "
          f"({r.pct_of_line:.0f}% of contributing lines)  |  {r.lines}")
    print(f"    signatures  UP : {r.sig_up}")
    print(f"    signatures  DN : {r.sig_dn}")
    print(f"    markers     UP : {r.mk_up}")
    print(f"    markers     DN : {r.mk_dn}")
    print(f"    KI67 = {r.KI67:+.2f}   members: {r.members}")

In [ ]:
# --- 7b.2 Automatic labelling from the signature profile --------------------------------------
# A label is assigned from the dominant signatures, then sanity-checked against markers.
def label_meta(sigp, mkp):
    top = sigp.sort_values(ascending=False)
    bot = sigp.sort_values()
    lead, lead_v = top.index[0], top.iloc[0]
    second = top.index[1]
    down = bot.index[0]
    parts = []
    if sigp["Proliferation"] > 3 and sigp["DNA_damage_response"] < 0:
        parts.append("cycling")
    if sigp["DNA_damage_response"] > 3 and sigp["Proliferation"] < 0:
        parts.append("damage-arrested")
    if sigp["EMT"] > 3 or sigp["Basal_B_claudin_low"] > 3:
        parts.append("mesenchymal")
    if sigp["Luminal"] > 3:
        parts.append("luminal")
    if sigp["Stem_CD44p_CD24n"] > 2.5:
        parts.append("stem-like")
    if sigp["Polycomb_repression"] > 3 or sigp["Bivalent_poised"] > 3:
        parts.append("Polycomb/bivalent")
    if sigp["Constitutive_heterochromatin"] > 3:
        parts.append("heterochromatic")
    if sigp["Enhancer_primed"] > 3:
        parts.append("enhancer-primed")
    if sigp["H3K4_promoter_vs_enhancer"] > 3:
        parts.append("promoter-weighted")
    if sigp["Open_chromatin"] > 3:
        parts.append("open-chromatin")
    if sigp["Elongation"] > 3:
        parts.append("elongation-high")
    if not parts:
        parts.append(f"{lead}-driven")
    return " / ".join(dict.fromkeys(parts))

ident["label"] = [label_meta(r["_sig"], r["_mk"]) for _, r in ident.iterrows()]
ident["cycle"] = np.where(ident["KI67"] > 0.25, "cycling",
                          np.where(ident["KI67"] < -0.25, "slow", "intermediate"))

print("=== Assigned identities ===")
show = ident[["meta", "n_lines", "n_cells", "label", "cycle", "lines"]]
print(show.to_string(index=False))

print("\n=== Conservation summary ===")
for n in sorted(ident["n_lines"].unique(), reverse=True):
    g = ident[ident.n_lines == n]
    tag = {5: "CONSERVED (all 5)", 4: "shared (4)", 3: "shared (3)",
           2: "restricted (2)", 1: "LINE-SPECIFIC"}[n]
    print(f"\n  {tag}:")
    for _, r in g.iterrows():
        print(f"    {r['meta']:<5} {r['label']:<55} [{r['lines']}]")

ident.drop(columns=["_sig", "_mk"]).to_csv(
    BASE / "outputs/permcell_scores/meta_archetype_identification.csv", index=False)
print(f"\nsaved -> outputs/permcell_scores/meta_archetype_identification.csv")

## 7c. UMAPs in marker space and archetype space

Two embeddings of the same 354,435 cells, each overlaid with **both** markers and signatures so the
representations can be compared directly:

* **Marker space** — the 29 z-scored markers. This is the raw measurement geometry.
* **Archetype space** — the per-cell **meta-archetype weight vector**. Each line was fitted
  separately, so archetype *k* in MCF7 is not archetype *k* in HCC70; the weights are therefore
  mapped onto the shared meta-archetypes by summing each line's archetype weights within their
  assigned meta. That produces one common coordinate system in which all five lines are directly
  comparable — which per-line $W$ on its own is not.

Both use the fit-on-subsample / transform-all approach (60,000-cell stratified fit), which is far
cheaper than building a neighbour graph over the full set and preserves global structure.

In [ ]:
# --- 7c.1 Build the three joint matrices --------------------------------------------------------
import umap

order_lines = DISP_ORDER
cell_line_vec = np.concatenate([np.repeat(l, MK[l].shape[0]) for l in order_lines])

# (a) marker space
X_marker = np.vstack([MK[l] for l in order_lines]).astype(np.float32)

# (b) signature space (already scaled the same way the models saw it)
X_sig = np.vstack([line_data[[k for k, v in LINE_DISP.items() if v == l][0]]["X"]
                   for l in order_lines]).astype(np.float32)

# (c) ARCHETYPE space: per-cell weight over the SHARED meta-archetypes.
meta_ids = sorted(arch_df["meta"].unique())
meta_cols = [f"M{m}" for m in meta_ids]
blocks = []
for l in order_lines:
    W = per_line_model[l]["W"]
    g = arch_df[arch_df.line == l]
    Wm = np.zeros((W.shape[0], len(meta_ids)), dtype=np.float32)
    for _, a_ in g.iterrows():
        Wm[:, meta_ids.index(a_.meta)] += W[:, a_.k - 1]
    blocks.append(Wm)
X_arch = np.vstack(blocks)

assert X_marker.shape[0] == X_sig.shape[0] == X_arch.shape[0] == len(cell_line_vec)
print(f"cells {X_marker.shape[0]:,} | marker {X_marker.shape[1]} | "
      f"signature {X_sig.shape[1]} | archetype(meta) {X_arch.shape[1]}")

dom_meta = np.array([meta_cols[i] for i in X_arch.argmax(1)])
print("\ndominant meta-archetype distribution:")
print(pd.Series(dom_meta).value_counts().sort_index().to_string())

In [ ]:
# --- 7c.2 Embeddings: fit on a stratified subsample, project everything ------------------------
UMAP_FIT_N, UMAP_CHUNK = 60_000, 50_000


def stratified_idx(labels, fit_n, seed=SEED):
    labels = np.asarray(labels); rng = np.random.default_rng(seed); n = len(labels)
    if fit_n >= n:
        return np.arange(n)
    parts = []
    for lab in np.unique(labels):
        pool = np.flatnonzero(labels == lab)
        take = max(1, int(round(fit_n * len(pool) / n)))
        parts.append(rng.choice(pool, size=min(take, len(pool)), replace=False))
    return np.sort(np.concatenate(parts))


def embed(rep, labels, label=""):
    rep = np.ascontiguousarray(np.asarray(rep, np.float32))
    fit = stratified_idx(labels, UMAP_FIT_N)
    red = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=SEED, verbose=False).fit(rep[fit])
    out = np.empty((len(rep), 2), np.float32)
    out[fit] = red.embedding_
    rest = np.setdiff1d(np.arange(len(rep)), fit)
    for s in range(0, len(rest), UMAP_CHUNK):
        b = rest[s:s + UMAP_CHUNK]
        out[b] = red.transform(rep[b])
    print(f"  [{label}] fitted on {len(fit):,} / projected {len(rep):,}", flush=True)
    return out


print("Computing UMAPs:")
UM = {
    "marker": embed(X_marker, cell_line_vec, "marker space (29 markers)"),
    "archetype": embed(X_arch, cell_line_vec, "archetype space (meta-archetype weights)"),
}
print("done")

In [ ]:
# --- 7c.3 Overview: cell line and dominant meta-archetype in both spaces -----------------------
def scatter(ax, emb, values, categorical, title, cmap="viridis", vmin=None, vmax=None, s=1.2):
    if categorical:
        cats = sorted(pd.unique(values))
        pal = sns.color_palette("tab20", len(cats))
        for c, col in zip(cats, pal):
            m = values == c
            ax.scatter(emb[m, 0], emb[m, 1], s=s, c=[col], alpha=0.5, rasterized=True, label=c)
    else:
        ax.scatter(emb[:, 0], emb[:, 1], s=s, c=values, cmap=cmap, alpha=0.6,
                   vmin=vmin, vmax=vmax, rasterized=True)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)


fig, axes = plt.subplots(2, 2, figsize=(15, 14))
for col, space in enumerate(["marker", "archetype"]):
    nice = "MARKER space (29 markers)" if space == "marker" else "ARCHETYPE space (meta weights)"
    scatter(axes[0, col], UM[space], cell_line_vec, True, f"{nice}\ncell line")
    scatter(axes[1, col], UM[space], dom_meta, True, f"{nice}\ndominant meta-archetype")
axes[0, 0].legend(markerscale=12, fontsize=9, loc="best", frameon=False)
axes[1, 1].legend(markerscale=12, fontsize=7, loc="best", ncol=2, frameon=False)
fig.suptitle("Same cells, two representations", fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
plt.savefig(PLOTS / "PermCell_UMAP_Overview.png", dpi=170, bbox_inches="tight")
plt.show()

In [ ]:
# --- 7c.4 MARKERS on both embeddings -----------------------------------------------------------
def grid_overlay(space, values_df, names, fname, suptitle, cmap="magma", q=(0.02, 0.98)):
    n = len(names); ncols = 6; nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.9, nrows * 2.9))
    axf = np.atleast_1d(axes).ravel()
    emb = UM[space]
    for i, nm in enumerate(names):
        v = values_df[:, i] if isinstance(values_df, np.ndarray) else values_df[nm].to_numpy()
        lo, hi = np.quantile(v, q[0]), np.quantile(v, q[1])
        scatter(axf[i], emb, v, False, nm, cmap=cmap, vmin=lo, vmax=hi, s=0.8)
    for ax in axf[n:]:
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=15, fontweight="bold", y=1.002)
    plt.tight_layout()
    plt.savefig(PLOTS / fname, dpi=140, bbox_inches="tight")
    plt.show()


grid_overlay("marker", X_marker, MARKERS_LIST, "PermCell_UMAP_Markers_on_MarkerSpace.png",
             "MARKERS on the MARKER-space UMAP")
grid_overlay("archetype", X_marker, MARKERS_LIST, "PermCell_UMAP_Markers_on_ArchetypeSpace.png",
             "MARKERS on the ARCHETYPE-space UMAP")

In [ ]:
# --- 7c.5 SIGNATURES on both embeddings --------------------------------------------------------
grid_overlay("marker", X_sig, SIG_NAMES, "PermCell_UMAP_Signatures_on_MarkerSpace.png",
             "PermCell SIGNATURES on the MARKER-space UMAP", cmap="RdBu_r")
grid_overlay("archetype", X_sig, SIG_NAMES, "PermCell_UMAP_Signatures_on_ArchetypeSpace.png",
             "PermCell SIGNATURES on the ARCHETYPE-space UMAP", cmap="RdBu_r")

# How well does each space separate the lines and the meta-archetypes?
from sklearn.neighbors import NearestNeighbors


def neighbour_purity(rep, labels, k=30, sample=20000, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    idx = rng.choice(len(rep), min(sample, len(rep)), replace=False)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(rep)
    _, ind = nn.kneighbors(rep[idx])
    return float((labels[ind[:, 1:]] == labels[idx][:, None]).mean())


p_s = pd.Series(cell_line_vec).value_counts(normalize=True).values
print("=== 30-NN purity in each representation (higher = more separated) ===")
print(f"  chance level for cell line: {np.sum(p_s**2):.3f}")
for nm, rep in [("marker space", X_marker), ("signature space", X_sig),
                ("archetype space", X_arch)]:
    print(f"  {nm:<18} cell-line purity {neighbour_purity(rep, cell_line_vec):.3f} | "
          f"meta-archetype purity {neighbour_purity(rep, dom_meta):.3f}")
print("\nCell-line purity near 1.0 means the space still separates lines rather than integrating")
print("them -- expected here, since every model was fitted per line by design.")

## 8. Summary

In [ ]:
print("=" * 78)
print("Archetypes in PermCell signature space -- summary")
print("=" * 78)
print("\n-- Inputs --")
print(f"  Signatures (discovery)     : {len(SIG_NAMES)}  (7 more held out for post-hoc use)")
print(f"  Score                      : raw (unsmoothed) permutation Z from sipsic_like_scores_v3")
print(f"  Input layer                : norm_divide, z-scored with one scaler across all 5 lines")
print(f"  Cells                      : {len(stacked):,}")
print(f"  Effective dimensionality   : {ev.sum()**2/(ev**2).sum():.2f} of {len(SIG_NAMES)}")

print("\n-- Per-line fits --")
for line in DISP_ORDER:
    m = per_line_model[line]
    s = sel_all[(sel_all.line == line) & (sel_all.K == m["K"])].iloc[0]
    print(f"  {line:<11} K={m['K']:<3} val_recon={s['val_best']:.4f}  stability={s['stability']:.3f}")
print(f"  Total archetypes: {TOTAL_ARCH}")

print("\n-- Cross-line matching --")
print(f"  Null threshold             : {MATCH_THRESHOLD:.3f}")
print(f"  Pairings above threshold   : {int(pairs_df['matched'].sum())}/{len(pairs_df)}")
_off = pair_scores.astype(float).where(~np.eye(len(DISP_ORDER), dtype=bool)).stack()
print(f"  Most / least similar pair  : {_off.idxmax()} {_off.max():.3f} | {_off.idxmin()} {_off.min():.3f}")

print("\n-- Meta-archetypes --")
print(f"  {TOTAL_ARCH} archetypes -> {len(meta_df)}")
for kind in order_k:
    g = meta_df[meta_df["kind"] == kind]
    if len(g):
        print(f"    {kind:<20} {len(g):>2}  ({', '.join('M'+str(m) for m in g['meta'])})")

print("\n-- Signature vs marker space --")
print(f"  mean stability : {cmp_tbl['stability_marker'].mean():.3f} (marker) -> "
      f"{cmp_tbl['stability_signature'].mean():.3f} (signature)")
print(f"  total K        : {cmp_tbl['K_marker'].sum()} -> {cmp_tbl['K_signature'].sum()}")
print("=" * 78)